# 02 — Silver projection

Projects the known v1 keys out of `bronze.plays_raw.payload` (VARIANT) into typed columns on `silver.plays`.

**Known-keys allow-list** — this is the explicit contract between bronze and silver. Anything in `payload` that isn't in this list is treated as **schema drift** by the detector in notebook 03.

We write silver with `mergeSchema=true` ahead of time so notebook 05 can widen it without recreating the table.

In [0]:
%pip install dotenv

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os
from dotenv import load_dotenv
load_dotenv()

try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv('UC_CATALOG', 'alexander_booth')
UC_SCHEMA     = os.getenv('UC_SCHEMA',  'hockey_schema_monitoring')
BRONZE_TABLE  = f'{UC_CATALOG}.{UC_SCHEMA}_bronze.plays_raw'
SILVER_TABLE  = f'{UC_CATALOG}.{UC_SCHEMA}_silver.plays'
KNOWN_KEYS_TBL = f'{UC_CATALOG}.{UC_SCHEMA}_silver.known_payload_keys'

print(f'Bronze : {BRONZE_TABLE}')
print(f'Silver : {SILVER_TABLE}')
print(f'Keys   : {KNOWN_KEYS_TBL}')

Bronze : alexander_booth.hockey_schema_monitoring_bronze.plays_raw
Silver : alexander_booth.hockey_schema_monitoring_silver.plays
Keys   : alexander_booth.hockey_schema_monitoring_silver.known_payload_keys


## Known keys (the bronze→silver contract)

We store the allow-list in a tiny UC table so the drift detector (notebook 03) and the schema-evolution step (notebook 05) can both reference it. Keeping it in UC also means the customer's data team can review schema additions in PRs against this table instead of buried in notebook code.

In [0]:
V1_KEYS = [
    'game_id', 'season', 'period', 'period_time',
    'event_type', 'team_abbrev', 'player_id',
    'x_coord', 'y_coord', 'event_ts',
]

from pyspark.sql import Row
(spark.createDataFrame([Row(key=k) for k in V1_KEYS])
      .write.mode('overwrite').saveAsTable(KNOWN_KEYS_TBL))

spark.sql(f'SELECT * FROM {KNOWN_KEYS_TBL} ORDER BY key').show(truncate=False)

+-----------+
|key        |
+-----------+
|event_ts   |
|event_type |
|game_id    |
|period     |
|period_time|
|player_id  |
|season     |
|team_abbrev|
|x_coord    |
|y_coord    |
+-----------+



In [0]:
# Typed projection of v1 keys.
# Silver is built with mergeSchema=true so notebook 05 can widen it in place.
spark.sql(f'''
    CREATE OR REPLACE TABLE {SILVER_TABLE}
    USING DELTA
    TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
    AS
    SELECT
        ingest_ts,
        source_version,
        payload:game_id::string      AS game_id,
        payload:season::string       AS season,
        payload:period::int          AS period,
        payload:period_time::string  AS period_time,
        payload:event_type::string   AS event_type,
        payload:team_abbrev::string  AS team_abbrev,
        payload:player_id::long      AS player_id,
        payload:x_coord::int         AS x_coord,
        payload:y_coord::int         AS y_coord,
        cast(payload:event_ts::string AS timestamp) AS event_ts
    FROM {BRONZE_TABLE}
''')

spark.sql(f'SELECT COUNT(*) AS rows FROM {SILVER_TABLE}').show()
spark.sql(f'DESCRIBE TABLE {SILVER_TABLE}').show(truncate=False)

+----+
|rows|
+----+
|3200|
+----+

+--------------+---------+-------+
|col_name      |data_type|comment|
+--------------+---------+-------+
|ingest_ts     |timestamp|NULL   |
|source_version|string   |NULL   |
|game_id       |string   |NULL   |
|season        |string   |NULL   |
|period        |int      |NULL   |
|period_time   |string   |NULL   |
|event_type    |string   |NULL   |
|team_abbrev   |string   |NULL   |
|player_id     |bigint   |NULL   |
|x_coord       |int      |NULL   |
|y_coord       |int      |NULL   |
|event_ts      |timestamp|NULL   |
+--------------+---------+-------+



In [0]:
spark.sql(f'''
    SELECT event_type, COUNT(*) AS n
    FROM {SILVER_TABLE}
    GROUP BY event_type
    ORDER BY n DESC
''').show()

+------------+---+
|  event_type|  n|
+------------+---+
|        goal|373|
|blocked-shot|366|
|         hit|363|
|shot-on-goal|361|
| missed-shot|354|
|    giveaway|353|
|     faceoff|350|
|     penalty|340|
|    takeaway|340|
+------------+---+

